In [2]:
import requests
import pandas as pd

headers = {"User-Agent": "Omid Firouzinia omidfiruzi0084@gmail.com"}
url = "https://www.sec.gov/files/company_tickers_exchange.json"

resp = requests.get(url, headers=headers)
print("Status:", resp.status_code)

Status: 200


In [3]:
data = resp.json()
sec = pd.DataFrame(data["data"], columns=data["fields"])

print(len(sec), "companies")
sec.head()

10461 companies


,cik,name,ticker,exchange
0,1045810,NVIDIA CORP,NVDA,Nasdaq
1,320193,Apple Inc.,AAPL,Nasdaq
2,1652044,Alphabet Inc.,GOOGL,Nasdaq
3,789019,MICROSOFT CORP,MSFT,Nasdaq
4,1018724,AMAZON COM INC,AMZN,Nasdaq


In [4]:
import json
from datetime import date

with open(f"data/raw/sec_company_tickers_exchange_{date.today()}.json", "w") as f:
    json.dump(data, f)

In [5]:
config = json.load(open("config.json"))

ours = sec[sec["ticker"].isin(config["tickers"])].copy()
print(len(ours), "of", len(config["tickers"]), "found")
print("Missing:", set(config["tickers"]) - set(ours["ticker"]))
ours.head()

59 of 59 found
Missing: set()


,cik,name,ticker,exchange
0,1045810,NVIDIA CORP,NVDA,Nasdaq
1,320193,Apple Inc.,AAPL,Nasdaq
2,1652044,Alphabet Inc.,GOOGL,Nasdaq
3,789019,MICROSOFT CORP,MSFT,Nasdaq
4,1018724,AMAZON COM INC,AMZN,Nasdaq


In [6]:
ours["cik"] = ours["cik"].astype(str).str.zfill(10)
ours["exchange"].value_counts()

exchange
NYSE      37
Nasdaq    22
Name: count, dtype: int64

In [7]:
candidates = pd.read_csv("data/candidate_stocks.csv")
check = ours.merge(candidates[["ticker", "expected_exchange"]], on="ticker")

print("Mismatches:", (check["exchange"] != check["expected_exchange"]).sum())

Mismatches: 0


In [8]:
ours["snapshot_date"] = date.today().isoformat()
ours.to_csv("data/processed/stock_metadata.csv", index=False)

In [9]:
pd.read_csv("data/processed/stock_metadata.csv", dtype={"cik": str})

,cik,name,ticker,exchange,snapshot_date
0,0001045810,NVIDIA CORP,NVDA,Nasdaq,2026-09-24
1,0000320193,Apple Inc.,AAPL,Nasdaq,2026-09-24
2,0001652044,Alphabet Inc.,GOOGL,Nasdaq,2026-09-24
3,0000789019,MICROSOFT CORP,MSFT,Nasdaq,2026-09-24
4,0001018724,AMAZON COM INC,AMZN,Nasdaq,2026-09-24
5,0001326801,"Meta Platforms, Inc.",META,Nasdaq,2026-09-24
6,0000059478,ELI LILLY & Co,LLY,NYSE,2026-09-24
7,0000019617,JPMORGAN CHASE & CO,JPM,NYSE,2026-09-24
8,0001403161,VISA INC.,V,NYSE,2026-09-24
9,0002115436,ExxonMobil Holdings Corp,XOM,NYSE,2026-09-24
